# CP6 - IA: Treinando uma LLM Local com GPU (Fine-tuning LoRA)

**Problema:** Assistente de perguntas e respostas (Q&A) em português sobre a FIAP e conceitos de Inteligência Artificial.

**Modelo base:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

**Técnica:** Fine-tuning com QLoRA (quantização 4-bits + LoRA)

## 1. Informações da GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print('CUDA disponivel:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Versao do CUDA (PyTorch):', torch.version.cuda)
    print('Memoria total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('ATENCAO: GPU nao detectada.')

## 2. Dependências

In [ ]:
!pip install -q -U transformers==4.44.2 datasets==2.21.0 peft==0.12.0 bitsandbytes==0.43.3 accelerate==0.33.0 trl==0.9.6

## 3. Dataset

In [ ]:
from google.colab import files
import os
if not os.path.exists('qa_fiap.jsonl'):
    uploaded = files.upload()

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='qa_fiap.jsonl', split='train')
print('Numero de exemplos:', len(dataset))
print('Exemplo:', dataset[0])

## 4. Formatação do dataset no template de chat

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = 'Voce e um assistente educacional da FIAP. Responda em portugues de forma clara e objetiva.'

def format_example(ex):
    pergunta = ex['instruction']
    if ex.get('input'):
        pergunta += '\n' + ex['input']
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': pergunta},
        {'role': 'assistant', 'content': ex['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {'text': text}

dataset = dataset.map(format_example)
print(dataset[0]['text'])

## 5. Modelo com quantização 4-bits (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Treinamento

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./tinyllama-fiap',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim='paged_adamw_8bit',
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
model.save_pretrained('tinyllama-fiap-lora')
tokenizer.save_pretrained('tinyllama-fiap-lora')
print('Modelo salvo em tinyllama-fiap-lora/')

## 7. Avaliação das respostas

In [ ]:
from transformers import pipeline

def responder(pergunta, max_new_tokens=200, temperature=0.7):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': pergunta},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    resposta = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return resposta.strip()

perguntas_teste = [
    'O que e a FIAP?',
    'O que e LoRA no treinamento de LLMs?',
    'Para que serve a GPU no treinamento de IA?',
    'O que e um Challenge na FIAP?',
    'Como organizar meu tempo de estudo?',
]

for p in perguntas_teste:
    print('='*70)
    print('PERGUNTA:', p)
    print('RESPOSTA:', responder(p))
    print()

## 8. Comparação de temperatura

In [ ]:
pergunta = 'O que e fine-tuning de um modelo de linguagem?'
for t in [0.2, 0.7, 1.0]:
    print('='*70)
    print(f'Temperatura = {t}')
    print(responder(pergunta, temperature=t))
    print()

## 9. Exportar o adaptador LoRA

In [ ]:
!zip -r tinyllama-fiap-lora.zip tinyllama-fiap-lora
from google.colab import files
files.download('tinyllama-fiap-lora.zip')